# B4 · Validation against Perseverance's real drive

**A rover is not a human.** Perseverance drives slopes and ground a suited astronaut would not walk, and
it avoids sand a human could cross. These checks only test that the DEM-derived slopes, our thresholds and
the planner behave sensibly on ground a vehicle really crossed. They are not proof that the walk is safe.

Inputs from the assets bundle (`ARIADNE_ASSETS`, default `~/ariadne-assets/`): `aoi.json`,
`bright-angel/slope_2m.f32`, `cost_2m.f32`, `track.geojson`. Planned legs come from the planner's own code
(`planner/app`). Thresholds come from `model/params.json`. Figures go to `notebooks/out/`.

In [ ]:
import json, math, os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "notebooks"), str(ROOT / "planner")]
import validation as v
from app import plan, routing

A = Path(os.environ.get("ARIADNE_ASSETS", Path.home() / "ariadne-assets")).expanduser()
B = A / "bright-angel"
OUT = ROOT / "notebooks" / "out"; OUT.mkdir(parents=True, exist_ok=True)
aoi = v.load_json(A / "aoi.json"); size = tuple(aoi["size_m"])
P = v.load_json(ROOT / "model" / "params.json")
caution, nogo = P["caution_slope_deg"]["value"], P["max_slope_deg"]["value"]
slope = v.load_f32(B / "slope_2m.f32", 2.0, size)
cost = v.load_f32(B / "cost_2m.f32", 2.0, size)
track = v.load_json(B / "track.geojson")
print("bundle", (A / "BUNDLE.txt").read_text().splitlines()[0] if (A / "BUNDLE.txt").exists() else "?",
      "| caution", caution, "° | no-go >", nogo, "°")

## 1. Slope along the rover track

Sampled every 0.5 m, weighted by path length; only track inside the AOI with slope data.

In [ ]:
s, l = v.sample_lines(v.track_lines(track), slope, 2.0, size)
stats = v.slope_stats(s, l, caution, nogo)
print(json.dumps(stats, indent=1))
fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.arange(0, math.ceil(s.max()) + 1, 1)
ax.hist(s, bins=bins, weights=l, color="#5b7a99")
for t, lab in ((caution, f"caution {caution}°"), (nogo, f"no-go > {nogo}°")):
    ax.axvline(t, color="#c0392b" if t == nogo else "#d4a017", lw=1.5)
    ax.text(t + 0.3, ax.get_ylim()[1] * 0.9, lab, fontsize=8)
ax.set_xlabel("slope on the 2 m grid (°)"); ax.set_ylabel("track length (m)")
ax.set_title(f"Perseverance track in the AOI ({stats['length_m']/1000:.2f} km): "
             f"{stats['go_share']:.1%} go, {stats['caution_share']:.1%} caution, {stats['nogo_share']:.1%} no-go", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "validation_slope_hist.png", dpi=120)

## 2. Planned route vs the rover's path

Same start and end: the rover's end-of-drive waypoints used as stops (`content/stops/index.json`). Rover path = track drives between the two sols.

In [ ]:
d = plan.load(A, A / "aoi.json")
PAIRS = [("tuff-cliff", "bright-angel", 1168, 1175), ("cheyava-falls", "masonic-temple", 1210, 1231)]
rows, paths = [], []
for a, b, sa, sb in PAIRS:
    pa, pb = d.stops[a], d.stops[b]
    res = routing.least_cost_path(d.costs, d.cost.px_m, d.cost.cell(*pa.xy), d.cost.cell(*pb.xy))
    planned = plan._points(d, res.cells, pa.xy, pb.xy)
    rover = v.chain(v.track_lines(track, (sa, sb)))
    rover_t, rover_nogo = v.time_along(rover, cost, 2.0, size)
    dv = v.divergences(planned, rover)
    rows.append({"leg": f"{a} -> {b}", "sols": f"{sa}-{sb}", "planned_m": round(v.length_m(planned)),
                 "rover_m": round(v.length_m(rover)), "ratio": round(v.length_m(planned) / v.length_m(rover), 2),
                 "planned_min": round(res.time_min, 1), "rover_min_in_model": round(rover_t, 1),
                 "rover_nogo_share": round(rover_nogo, 3), "end_gaps_m": [round(math.dist(rover[0], pa.xy)), round(math.dist(rover[-1], pb.xy))],
                 "divergences": dv})
    paths.append((a, b, planned, rover))
for r in rows: print(json.dumps(r))

In [ ]:
from matplotlib.colors import LightSource
dem = v.load_f32(B / "dem_4m.f32", 4.0, size)
hs = LightSource(azdeg=315, altdeg=35).hillshade(np.nan_to_num(dem, nan=np.nanmean(dem)), vert_exag=2, dx=4, dy=4)
fig, axes = plt.subplots(1, len(paths), figsize=(11, 5))
for ax, (a, b, planned, rover), r in zip(np.atleast_1d(axes), paths, rows):
    allp = np.vstack([planned, rover]); pad = 80
    x0, x1 = allp[:, 0].min() - pad, allp[:, 0].max() + pad; y0, y1 = allp[:, 1].min() - pad, allp[:, 1].max() + pad
    ax.imshow(hs, cmap="gray", extent=[0, size[0], 0, size[1]])
    ax.plot(rover[:, 0], rover[:, 1], color="white", lw=2.2, label=f"rover, sols {r['sols']} ({r['rover_m']} m)")
    ax.plot(planned[:, 0], planned[:, 1], color="#e67e22", lw=2, label=f"planned ({r['planned_m']} m, {r['planned_min']} min)")
    for dv in r["divergences"]:
        ax.annotate(f"{dv['max_off_m']:.0f} m off", dv["from_xy"], fontsize=7, color="#e67e22",
                    bbox=dict(boxstyle="round,pad=0.1", fc="black", ec="none", alpha=0.5))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_title(r["leg"], fontsize=9)
    ax.legend(fontsize=7, loc="lower left"); ax.set_xlabel("x east (m)"); ax.set_ylabel("y north (m)")
fig.tight_layout(); fig.savefig(OUT / "validation_route.png", dpi=110)